# FLUX.1-schnell on free Colab (T4) — sample

Generate images with **FLUX.1-schnell** on a **free** Colab T4 GPU. Compute is free; you just need a free Hugging Face token (the model is license-gated).

- Model: `black-forest-labs/FLUX.1-schnell` (**gated** — needs a free HF token + one-click license accept)
- Fits in 16GB T4 via `enable_model_cpu_offload()`
- ~4 inference steps; a T4 render takes roughly **30–90s** per image

**Before you run:** `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print('No GPU! Set Runtime -> Change runtime type -> T4 GPU')

## 2. Install dependencies
Takes ~1–2 min the first time.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf

## 3. Log in to Hugging Face
FLUX.1-schnell is license-gated, so you need a **free** token:
1. Create a **Read** token: https://huggingface.co/settings/tokens
2. Accept the license (one click): https://huggingface.co/black-forest-labs/FLUX.1-schnell
3. Run the cell below and paste the token when prompted.

## 4. Load FLUX.1-schnell
`enable_model_cpu_offload()` streams weights between CPU and GPU so it fits the T4's 16GB.
First run downloads ~24GB of weights (a few minutes on Colab's fast link).

In [ ]:
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16,
)
# Sequential offload keeps VRAM under the T4's 16GB and avoids the
# "tensors on cuda:0 and cpu" device-mismatch bug. Slower but robust.
# On a bigger GPU (>=24GB) replace both offload lines with: pipe.to("cuda")
pipe.enable_sequential_cpu_offload()
print("Pipeline ready.")

## 5. Generate an image
Edit `prompt` and re-run this cell as many times as you like.

## 4. Generate an image
Edit `prompt` and re-run this cell as many times as you like.

## 6. (Optional) Batch a few prompts
Generate several stills at once — the kind of b-roll the `tennessee-bound` pipeline consumes.

## 5. (Optional) Batch a few prompts
Generate several stills at once — the kind of b-roll the `tennessee-bound` pipeline consumes.

In [ ]:
prompts = [
    "aerial drone shot of a winding mountain road through autumn forest",
    "close-up of an old acoustic guitar on a wooden porch, warm light",
    "a country music stage at night, stage lights, crowd silhouettes",
]

for i, p in enumerate(prompts):
    img = pipe(
        p, guidance_scale=0.0, num_inference_steps=4,
        height=1024, width=1024, max_sequence_length=256,
        generator=torch.Generator("cpu").manual_seed(i),
    ).images[0]
    img.save(f"broll_{i}.png")
    print(f"saved broll_{i}.png  ::  {p}")
print("Done. Download from the Files panel on the left.")

---
### Notes
- **Free tier limits:** Colab may disconnect after idle/long sessions and the T4 isn't guaranteed at peak times. Downloaded weights and outputs are wiped when the runtime resets.
- **Want higher quality?** `FLUX.1-dev` (gated — needs a free HF token + license accept) gives better results but needs ~24GB, so use a paid A100/L4 runtime or a rented 4090.
- **Next step for volume:** move this to a single RTX 4090 (~$0.40/hr) and drop `enable_model_cpu_offload()` for full speed — thousands of images per $10.